In [1]:
import torch 
import os
import numpy as np
import torch.nn.functional as F


from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf100_test_data = datasets.CIFAR100(root = 'data', train = False, download= True, transform = transforms.ToTensor())

In [4]:
# create validation set
print(len(cf10_training_data))


print(len(cf100_training_data))

cf10_val_data = random_split(cf10_training_data, [int(len(cf10_training_data)*0.8), int(len(cf10_training_data)*0.2)])
#cf100_val_data = random_split(cf100_training_data, [int(len(cf100_training_data)*0.8), int(len(cf100_training_data)*0.2)])

50000
50000


In [5]:
training_loader = torch.utils.data.DataLoader(cf10_training_data, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_data, batch_size=32, shuffle=True)
testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

In [17]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1= nn.Conv2d(3, 6, 5,1)
        self.conv2 = nn.Conv2d(6, 16, 5,1)

        self.flatten = nn.Flatten()
        self.maxpool = nn.MaxPool2d(2,2)
        self.relu = nn.ReLU()
        

        self.fc1 = nn.Linear(16* 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10)

        self._init_weight()

    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        
    def forward(self,x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool()

        x = self.conv2(x)
        x = self.relu(x)
        x = self.maxpool()

        x = x.view(-1, 16*5*5)


        x = self.fc1(x)
        x = self.relu(x)

       
        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)
        return nn.log_softmax(x, dim = 1)

    

In [11]:
lenet = LeNet().to(device)
print(lenet)

LeNet(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [12]:
criterion  = nn.CrossEntropyLoss()
optimizer =torch.optim.Adam(lenet.parameters(), lr = 1e-3)


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def training_model(model, nr_epochs):
    losses = []
    lenet.to(device)
    model.train()
    

    for epoch in range(nr_epochs):
        running_loss = 0.0
        for i , data in enumerate(training_loader):
            inputs, label = data
            inputs = inputs.to(device)
            label = label.to(device)

            optimizer.zero_grad()

            outputs = model.forward(inputs)
            loss = criterion(outputs, label)

            

            

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(training_loader)
        print(f"Epoch {epoch}, Loss : {epoch_loss}")

            

    # return losses


training_model(lenet, 20)

TypeError: MaxPool2d.forward() takes 2 positional arguments but 3 were given